In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

In [0]:
dbutils.widgets.text("Incremental Flag","0")

In [0]:
increm_flag=dbutils.widgets.get("Incremental Flag")
print(increm_flag)

In [0]:
df_src=spark.sql('''
                 select * from parquet.`abfss://silver@healthcaredatalakedeb.dfs.core.windows.net/hr_provider`''')

In [0]:
df_src.limit(5).display()

In [0]:
if spark.catalog.tableExists("healthcare.gold.dim_provider"):
    df_sink=spark.sql('''
                    select * from healthcare.gold.dim_provider''')
else:
    df_sink=spark.sql('''
            select 1 as dim_provider_key,npi,provider_name,specialty,department,facility_id,provider_type,
            CAST(NULL AS String) as record_hash,
            CAST(NULL AS Date) as effective_start_date,
            CAST(NULL AS Date) as effective_end_date,
            CAST(NULL AS String) as is_current,
            CAST(NULL AS Timestamp) as load_timestamp from
            parquet.`abfss://silver@healthcaredatalakedeb.dfs.core.windows.net/hr_provider` where 1=0
            ''')

In [0]:
df_sink.limit(5).display()

## Generate Hash value for the incoming source data

In [0]:
tracked_cols=["provider_name","specialty","department","facility_id","provider_type"]
df_src = df_src.withColumn("record_hash",sha2(concat_ws("||", *tracked_cols), 256))

In [0]:
df_sink_current=df_sink.filter(df_sink["is_current"]=="Y")
df_filter=df_src.join(df_sink_current,df_src["npi"]==df_sink_current["npi"],"left").select(df_src["*"],
df_sink_current["dim_provider_key"],df_sink_current["record_hash"].alias("sink_record_hash"))

In [0]:
df_filter.limit(5).display()

## New records

In [0]:
df_new_record=df_filter.filter(df_filter["dim_provider_key"].isNull())

In [0]:
df_new_record.limit(5).display()

## Changed Records

In [0]:
df_changed_record=df_filter.filter(df_filter["dim_provider_key"].isNotNull() & df_filter["sink_record_hash"]!=df_filter["record_hash"])

In [0]:
df_changed_record = df_filter.filter(
    df_filter["dim_provider_key"].isNotNull()
    & (df_filter["sink_record_hash"] != df_filter["record_hash"])
)
df_changed_record.limit(5).display()

## Unchanged records

In [0]:
df_unchanged_record=df_filter.filter(df_filter["dim_provider_key"].isNotNull() 
                                     & (df_filter["sink_record_hash"] == df_filter["record_hash"]))

In [0]:
df_unchanged_record.limit(5).display()

In [0]:
df_insert=df_new_record.unionByName(df_changed_record).withColumn("isNewProvider",col("dim_provider_key").isNull())
df_insert.limit(5).display()

In [0]:
if increm_flag=='0':
    max_value=1
else:
    max_value_df=spark.sql('''
                select coalesce(MAX(dim_provider_key),0) from healthcare.gold.dim_provider''')
    max_value=max_value_df.collect()[0][0]+1

In [0]:
windowSpec=Window.orderBy(col("npi"))
df_insert=df_insert.withColumn(
    "dim_provider_key",row_number().over(windowSpec)+lit(max_value-1))\
    .withColumn("effective_start_date",when(col("isNewProvider"),to_date(lit("1900-01-01"))).otherwise(current_date()))\
        .withColumn("effective_end_date",to_date(lit("9999-12-31")))\
            .withColumn("is_current",lit("Y"))\
                .withColumn("load_timestamp",lit(current_timestamp()))
df_insert.limit(5).display()


In [0]:
df_insert=df_insert.drop("sink_record_hash")\
    .drop("isNewProvider")
df_insert.limit(5).display()

In [0]:
from delta.tables import DeltaTable
if spark.catalog.tableExists("healthcare.gold.dim_provider"):
    deltaTable=DeltaTable.forName(spark,"healthcare.gold.dim_provider")
    #insert new record and new version of old record
    df_insert.write.format("delta").mode("append").saveAsTable("healthcare.gold.dim_provider")
    # make the old record inactive
    deltaTable.alias("target").merge(df_changed_record.alias("source"),"target.dim_provider_key=source.dim_provider_key")\
        .whenMatchedUpdate(
            set={
                "is_current": lit("N"),
                "effective_end_date": lit(current_date())
            }
        )\
            .execute()
else:
    df_insert.write.format("delta")\
        .mode("append")\
            .saveAsTable("healthcare.gold.dim_provider")


In [0]:
%sql
select * from healthcare.gold.dim_provider;